# Notebook of the data section

**All plots in the report can be reproduced here**

To save the plots as pgf figures, you will need a working latex distribution
with those apt packages installed

```bash
sudo apt install dvipng cm-super texlive-fonts-recommended
!sudo apt-get install texlive-latex-recommended
!sudo apt install texlive-latex-extra
!sudo apt install texlive-xetex
!sudo apt install ttf-mscorefonts-installer
```


In [ ]:
from pathlib import Path

import wandb
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
from fm_benchmark_remote_sensing.data.pastis_r_preview_dataset import (
    PastisEmbeddingPreviewDataset,
)
import contextily as cx
import geopandas as gpd


In [ ]:
plt.style.use(["science", "grid", "ieee", "pgf"])
plt.rcParams.update(
    {
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "pgf.preamble": "\n".join(
            [
                r"\usepackage{fontspec}",
                r"\usepackage{unicode-math}",
                r"\setmainfont{Times New Roman}",
                r"\usepackage{siunitx}",
            ]
        ),
    }
)


In [ ]:
wandb.login()


In [ ]:
api = wandb.Api()

dataset_artifact = api.artifact(
    "maxerbox-org/benchmark-fm-models-remote-sensing/preview-pertuis-pastis-r-embeddings:latest",
    type="dataset",
)

preview_root = Path("../data/preview-pertuis-pastis-r-embeddings")
if not preview_root.exists():
    dataset_artifact.download(root=preview_root)

dataset = PastisEmbeddingPreviewDataset(
    pastis_r_root=preview_root / "PASTIS_R",
    ae_root=preview_root / "AE_EMBEDDING",
    tessera_root=preview_root / "TESSERA_EMBEDDING",
)

In [ ]:
dataset[0].tessera_embedding.shape

In [ ]:
global_metadata = gpd.read_file(preview_root / "metadata.geojson")
global_metadata = global_metadata.to_crs(epsg=3857)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

global_metadata.plot(
    ax=ax,
    facecolor="none",  # transparent fill
    edgecolor="yellow",  # strong color
    linewidth=2,
    zorder=3,
)
plt.title("PASTIS-R dataset preview")
plt.grid(False)
cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.savefig("map.pdf", bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

dataset.pastis_dataset.meta_patch.plot(
    ax=ax,
    facecolor="none",  # transparent fill
    edgecolor="yellow",  # strong color
    linewidth=2,
    zorder=3,
)
plt.title("PASTIS-R dataset subset")
plt.grid(False)
cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.savefig("../report/figures/subset_preview_pastis_r.pdf", backend="pgf")
